# L2a: Introduction to Functions and Interfaces

Functions turn a mathematical rule into a callable interface. We will progress from a direct Fibonacci function to typed models and multiple dispatch, while making input and mutation behavior explicit.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Define a function as an interface:__ Write a documented Julia function with typed arguments and a declared return type, so that a caller can use it correctly without reading its body.
> * __Handle base cases and mutation deliberately:__ Use early returns to keep base cases out of the main algorithm, and distinguish a function that returns a new value from one that modifies its argument in place.
> * __Select implementations with multiple dispatch:__ Use the types of the arguments to choose among several implementations of the same operation, rather than branching on type inside one function.

Let's get started!
___

## Setup, Data, and Prerequisites

Run the local setup cell first. It activates the single pinned course environment, loads every package used by this meeting, and includes any meeting source code.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

Besides Julia's `Base` library, `Include.jl` loads [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/), which the rest of the week uses. This lecture checks its own results with [the `@assert` macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert) instead.

___

## Anatomy of a Function
A mathematical function $f:A\rightarrow B$ maps inputs in a domain $A$ to outputs in a codomain $B$. In a program, a function is a named block of code that can receive inputs and perform a task. It may return a value, produce a side effect such as printing or modifying data, or do both.

Every function definition has the same core pieces: a **name**, **parameters**, a **body**, and **return behavior**. The syntax varies by language:

| Language | Definition | Body | Return and types |
|:--|:--|:--|:--|
| **Julia** | `function f(x)` ... `end`, or `f(x) = expression` | Delimited by `end` | `return` is optional; the last expression is returned. Type annotations are optional. |
| **MATLAB/Octave** | `function y = f(x)` ... `end` | Delimited by `end` | Return values are named in the definition and assigned in the body. Types are dynamic. |
| **Python** | `def f(x):` | Indented | Values are returned with `return`. Type hints are optional and are not enforced at runtime. |
| **C** | `ReturnType f(ArgumentType x)` | Enclosed in `{}` | Argument and return types are declared; a non-`void` function returns a compatible value. |

The companion [Julia](examples/function_anatomy.jl), [Python](examples/function_anatomy.py), [C](examples/function_anatomy.c), and [Octave](examples/fahrenheit_to_celsius.m) files implement the same temperature-conversion function. See the [examples README](examples/README.md) for command-line and REPL instructions.

Now let's examine these pieces in Julia.
___

## Example: Starter Fibonacci Sequence Function
Let's look at a simple implementation of a function that computes the Fibonacci sequence given an integer $n\in\mathbb{Z}_{\geq{0}}$ as input. A [Fibonacci sequence](https://en.wikipedia.org/wiki/Fibonacci_sequence) is composed of the Fibonacci numbers $F_{n}$:
$$
\begin{align*}
F_{0} & = 0 \quad n = 0\\
F_{1} & = 1 \quad n = 1\\
F_{n} & = F_{n-2} + F_{n-1}\quad{n\geq{2}}
\end{align*}
$$

Here's an example function that computes the Fibonacci numbers in Julia. It starts [with the function documentation (don't forget the documentation!)](https://docs.julialang.org/en/v1/manual/documentation/#Writing-Documentation), then defines the function name and parameters, and finally implements the logic to compute the Fibonacci number and returns the result.

In [ ]:
"""
    fibonacci(n::Int64) -> Dict{Int64, Int64}

Compute the Fibonacci sequence up to the nth element and return it as a dictionary.

### Arguments
- `n::Int64`: The index of the Fibonacci sequence to compute up to (must be a non-negative integer).

### Returns
- `Dict{Int64, Int64}`: A dictionary where keys are indices and values are the corresponding Fibonacci numbers.
"""
function fibonacci(n::Int64)::Dict{Int64, Int64} # I always use type annotations, it helps with debugging and understanding the code
    
    # Error condition: If n is negative, throw an error
    # We'll use an assertion here, there are better ways to handle this, but this is simple and effective (for now)
    @assert n >= 0 "Hmmm. Major Malfunction - the argument `n` must be a non-negative integer";

    # initialize -
    sequence = Dict{Int64, Int64}(); # create an empty dictionary to hold the Fibonacci numbers

    # Early return pattern for n = 0 or n = 1
    # This is a common pattern in programming, especially for recursive functions (which this is not, but we'll get to that later)
    # We implement this using an if-else statement (we'll dig a bit deeper into this later)
    if n == 0 
        sequence[0] = 0;
        return sequence;
    elseif n == 1
        sequence[0] = 0;
        sequence[1] = 1;
        return sequence;
    end

    # So if we get here, we know n >= 2!

    # initialize the first two Fibonacci numbers
    sequence[0] = 0;
    sequence[1] = 1;

    # main loop, compute F₂, ....
    test_range = range(2, stop=n, step=1) |> collect; # what is this doing? It's creating a range from 2 to n, with a step of 1, and then converting it to an array (or vector) using `collect`. 
    for i ∈ test_range # what is this short-hand for?
        sequence[i] = sequence[i-1] + sequence[i-2]
    end

    # return the populated sequence dictionary to the caller
    return sequence;
end;

Call the function with an integer argument $n\in\mathbb{Z}_{\geq{0}}$, and store the result in a `fibonacci_dictionary::Dict{Int,Int}` variable. This dictionary maps the integers to their corresponding Fibonacci numbers.

In [ ]:
fibonacci_dictionary = fibonacci(10) # n = 10, seems reasonable

__Check__: Starting at 0, we know that the Fibonacci numbers are 0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, ... . Let's check these values against our results stored in the `fibonacci_dictionary::Dict{Int64,Int64}` dictionary using [the `@assert` macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert).

> __Interesting__: We have multiple conditions to check, i.e., the computed values for $F_{0},F_{1},\dots,F_{10}$ are correct. We've implemented this by enclosing [the `@assert` macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert) in a `for` loop. This is a common pattern, but not the only (or most efficient) way we could have done this.

Do we pass the tests?

In [ ]:
let 
    
    # initialize -
    true_results = [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55]; # F_{0} -> F_{10}, indexing is 1-based!

    # Loop based implementation to check multiple results
    # check the results
    for i ∈ 0:10 # what??
        @assert fibonacci_dictionary[i] == true_results[i+1] "Fibonacci number at index $i is incorrect"
    end


    println("All tests passed!")
end

What happens if we pass a negative integer to the function? The [`@assert` macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert) in our implementation should reject it and throw [an `AssertionError`](https://docs.julialang.org/en/v1/base/base/#Core.AssertionError).

> __Why catch it here?__ An uncaught error stops the notebook at this cell. A stack trace, the report of the sequence of calls leading up to an error, is worth learning to read, but we want the rest of the notebook to run. So we wrap the call in [a try-catch block](https://docs.julialang.org/en/v1/base/base/#try) and capture the message as a string instead.

What does the error say?

In [ ]:
expected_error = try
    fibonacci(-12)
    "no error"
catch error
    sprint(showerror, error)
end

Capturing the message is one use of `try-catch`. The other is __recovery__: carrying on with a fallback when the call fails. Below we try to assign the result to `fibonacci_dictionary_2`, and the program keeps running past the failure rather than stopping:

In [ ]:
fibonacci_dictionary_2 = Dict{Int64,Int64}(); # initialize an empty dictionary to hold the Fibonacci numbers
try
    fibonacci_dictionary_2 = fibonacci(-10) # n = -10. This should throw an error
catch e
    println("Caught an error: $e")
end
println("Program continues after handling the error.")

So what is in `fibonacci_dictionary_2` now? The call threw before it could return a value, so the assignment never happened and the variable still holds the empty dictionary it was initialized with. The type is intact; the contents are not:

In [ ]:
fibonacci_dictionary_2 |> typeof

___

## Example: Complex Fibonacci Sequence Function
Functions can be more complex, with multiple parameters, different types of return values, and even side effects. Let's extend our Fibonacci function to take composite types, optional parameters, and reimagine how we structure the code by introducing a key concept called encapsulation.

To start, let's define several composite types to represent the Fibonacci sequence and its properties, and how we wish to implement the function.

In [ ]:
abstract type AbstractSequenceModel end # This is an abstract type for sequence models
abstract type AbstractIterationModel end # This is an abstract type for iteration models

"""
    MyFibonacciSequenceModel <: AbstractSequenceModel

A mutable struct to represent a Fibonacci sequence. 

### Fields
- `n::Int64`: The largest index to compute, so the sequence runs from `F₀` to `Fₙ` and holds `n + 1` entries.
- `sequence::Dict{Int64, Int64}`: The sequence itself, stored as a dictionary with indices as keys and Fibonacci numbers as values.

"""
mutable struct MyFibonacciSequenceModel <: AbstractSequenceModel

    # data -
    n::Int64 # number of elements in the sequence
    sequence::Dict{Int64, Int64} # the sequence itself

    # constructor -
    MyFibonacciSequenceModel() = new();
end

"""
    MyForLoopIterationModel <: AbstractIterationModel

An immutable struct to represent a for loop iteration model. This type has no fields and serves as a marker for a for loop iteration implementation.
"""
struct MyForLoopIterationModel <: AbstractIterationModel
    MyForLoopIterationModel() = new();
end

"""
    MyWhileLoopIterationModel <: AbstractIterationModel

An immutable struct to represent a while loop iteration model. This type has no fields and serves as a marker for a while loop iteration implementation.
"""
struct MyWhileLoopIterationModel <: AbstractIterationModel
    MyWhileLoopIterationModel() = new();
end

Next, define a public `fibonacci!(...)` method that validates inputs and manages the state of `MyFibonacciSequenceModel`. It delegates the calculation to internal `_fibonacci(...)` methods.

Julia does not enforce private methods. A leading underscore (`_`) marks a method as internal by convention. Separating the public interface from the calculation allows the implementation to change without changing how callers use the function.

> __Method versus function__: A __function__ is a named operation. A __method__ is one implementation of that function for a particular set of argument types. With multiple dispatch, Julia selects a method using the types of the arguments.

The next cells implement this design.

In [ ]:
# -- PRIVATE METHODS BELOW HERE ------------------------------------------------------------------------------------------------------- #
function _fibonacci(sequencemodel::MyFibonacciSequenceModel, iterationmodel::MyForLoopIterationModel)

    @info "Debug message: We are using the for loop iteration model"

    # initialize -
    n = sequencemodel.n;
    sequence = Dict{Int64, Int64}();

    # base cases: same early-return pattern as the starter function above.
    # Without the n == 0 guard we hand back an F₁ the caller never asked for, and
    # disagree with the while-loop method below, which gets this right.
    sequence[0] = 0;
    if (n == 0)
        sequencemodel.sequence = sequence;
        return nothing;
    end
    sequence[1] = 1;

    # main loop, compute F₂, ....
    for i ∈ 2:n # what is this short-hand for?
        sequence[i] = sequence[i-1] + sequence[i-2]
    end

    # update the model -
    sequencemodel.sequence = sequence;
    return nothing; # this method mutates its argument; it does not return the sequence
end

function _fibonacci(sequencemodel::MyFibonacciSequenceModel, iterationmodel::MyWhileLoopIterationModel)

    @info "Debug message: We are using the while loop iteration model"

    # check: is n legit?
    n = sequencemodel.n;
    sequence = Dict{Int64, Int64}();
    
    # main loop 
    should_loop_continue = true
    i = 0;
    while (should_loop_continue == true)
       
        # conditional logic: hardcode 0, 1 else gets all other cases
        if (i == 0)
            sequence[i] = 0; 
        elseif (i == 1)
            sequence[i] = 1;
        else
            sequence[i] = sequence[i - 1] + sequence[i - 2]
        end

        # update i -
        i += 1; # this is short-hand for i = i + 1

        # check: should we go around again?
        if (i>n)
            should_loop_continue = false;
        end
    end
    
    # update the model -
    sequencemodel.sequence = sequence;
    return nothing; # this method mutates its argument; it does not return the sequence
end
# -- PRIVATE METHODS ABOVE HERE ------------------------------------------------------------------------------------------------------- #

# -- PUBLIC METHODS BELOW HERE -------------------------------------------------------------------------------------------------------- #
"""
    function fibonacci!(sequencemodel::MyFibonacciSequenceModel; 
        iterationmodel::T = MyForLoopIterationModel()) where T <: AbstractIterationModel

This function computes the Fibonacci sequence, given sequence and iteration models. 
The sequence model is updated in place (the sequence model is mutable). 
The iteration model is used to determine the type of loop to use.

# Arguments
- `sequencemodel::MyFibonacciSequenceModel`: The sequence model to update. The sequence model must have a field `n::Int64` giving the largest index to compute.
- `iterationmodel::T`: The iteration model to use. It must be a subtype of `AbstractIterationModel`. The default is `MyForLoopIterationModel`.

There is no return value. The `sequencemodel` is updated in place.
"""
function fibonacci!(sequencemodel::MyFibonacciSequenceModel; 
    iterationmodel::T = MyForLoopIterationModel()) where T <: AbstractIterationModel
    
    # Error condition: If n is negative, throw an error
    # We'll use an assertion here, there are better ways to handle this, but this is simple and effective (for now)
    @assert sequencemodel.n >= 0 "Hmmm. Major Malfunction - the argument `n` must be a non-negative integer";

    # Status: If we get here, then we know n >= 0
    _fibonacci(sequencemodel, iterationmodel); # multiple dispatch to the appropriate implementation
    return nothing;
end;

Let's start by building the composite types to represent the Fibonacci sequence and its properties. We'll define a `MyFibonacciSequenceModel` type to hold the Fibonacci numbers; let's save this in the `my_sequence_model::MyFibonacciSequenceModel` variable.

In [ ]:
my_sequence_model = let
    
    # initialize -
    n = 10; # number of elements to compute
    model = MyFibonacciSequenceModel(); # create an empty model

    # When the model is empty, the fields are undefined. We cannot access them until we set them!
    model.n = n; # set the number of elements to compute
    model.sequence = Dict{Int64, Int64}(); # initialize the sequence dictionary

    # return the model to the caller
    model
end

Hold on to something, this next part is going to get weird! The sequence field of the `MyFibonacciSequenceModel` model should be initialized to an empty dictionary of type `Dict{Int,Int}`, i.e., its length (number of things it holds) should be `0`. Is this true?

In [ ]:
@assert length(my_sequence_model.sequence) == 0

Ok, so now let's call our public `fibonacci!(...)` method and see what happens:

In [ ]:
fibonacci!(my_sequence_model); # what is going to happen?

That seems to have worked, but how do we get the results (the public `fibonacci!(...)` method doesn't return anything)? Also, why is there a `!` at the end of the function name? 
* _Mutating methods_: In Julia, a `!` at the end of a function name indicates that the function _modifies its arguments_ in some way that will be visible after the method execution has ended. In this case, the `fibonacci!` method modifies the `my_sequence_model` by populating its `sequence` field with Fibonacci numbers.
* _Is there something magical about the `!` character_? No, adding the `!` character to the end of a function name is _not magic_. It's just a convention to help identify functions that may change state or data outside the local scope of the function. In this particular case, the `fibonacci!` method modifies the `my_sequence_model` by populating its `sequence` field with Fibonacci numbers.

If that is true, we should see the results stored in the `my_sequence_model.sequence` variable. Let's check this by computing the length of the sequence field, and checking that it is larger than `0` (i.e., it has some values in it).

In [ ]:
@assert length(my_sequence_model.sequence) > 0

The other interesting thing from that call was that [the `@info` message](https://docs.julialang.org/en/v1/stdlib/Logging/#Logging.@logmsg) for the for loop got called. We didn't provide an implementation type, how did that work?

> __Optional keyword arguments__: The `iterationmodel::T` optional argument defaults to an instance of `MyForLoopIterationModel`. If we wanted to use our while loop implementation, should we pass in a `MyWhileLoopIterationModel` instance?

Let's check this out by calling the `fibonacci!` method again, but this time passing in a `MyWhileLoopIterationModel` instance as the optional argument. We should see the debug message for the while loop get called instead of the one for the for loop.

In [ ]:
fibonacci!(my_sequence_model, iterationmodel = MyWhileLoopIterationModel()); # what is going to happen?

In [ ]:
my_sequence_model.sequence # check the result

### Arguments and multiple dispatch
The calls above reach the same public `fibonacci!(...)` method, but `_fibonacci(...)` uses either the `for` or `while` implementation. How does Julia choose between them?

> __Multiple dispatch__: Julia selects a method using the runtime types of all positional arguments. The type of `iterationmodel` therefore selects the matching `_fibonacci(...)` method.

Julia supports three common argument forms:
* __Positional arguments__ are supplied in the order defined by the function signature. The caller must provide the required number of values, and their types must match an applicable method.
* __Optional positional arguments__ occupy a fixed position but have a default value. A caller may omit the argument to use the default or provide a value in that position to override it.
* __Keyword arguments__ appear after a semicolon in the signature and are passed by name. They can have default values, and naming them at the call site makes the purpose of each value clear without relying on argument order.

Keyword arguments do __not__ participate in dispatch, so two methods cannot differ only in the type of a keyword argument. `iterationmodel` is a keyword argument to `fibonacci!(...)`, but the public method passes it as a positional argument to `_fibonacci(...)`. Dispatch occurs on that second call.

See the [Julia methods documentation](https://docs.julialang.org/en/v1/manual/methods/#Methods) for more details.
___

## Looking ahead to Lab
In Lab `L2b` we continue to explore these ideas with a function that runs, raises no errors, but still returns the wrong answer. 

___

## Summary
A function is not just a rule for computing a value; it is an interface, and its argument types, return type, errors, and mutation behavior are all part of what it promises.

> __Key Takeaways:__
>
> * **The signature is the contract:** Argument types, the declared return type, the errors a function raises, and whether it mutates its input are the parts a caller depends on, so they deserve as much thought as the algorithm.
> * **Early returns keep base cases visible:** Handling the trivial inputs first and returning immediately leaves the main body free to express the general case, which is easier to read and easier to test.
> * **Dispatch replaces type branching:** Julia selects a method from the types of the arguments, so alternative implementations of one operation can live in separate methods rather than in one function full of type checks.

The functions you write from here on are the units the rest of the course composes: labs test them, later weeks import them, and problem sets extend them.
___